# AITR Lab 02: Evaluating Tool-Selection Accuracy in Multi-Tool Agentic AI
**Author**: Bharat Chouhan (MHM2026005)  
**Topic ID**: 12  

This interactive notebook reproduces the key experimental findings:
1. **Worked Numerical Example Verification (Section 5.7)**
2. **Catalogue Semantic Overlap Degradation (RQ1 & RQ2)**
3. **Horizon Compounding Error Collapse (RQ3)**

In [ ]:
def compute_rpi(actual_tools, pred_tools):
    set_a, set_p = set(actual_tools), set(pred_tools)
    return len(set_a & set_p) / len(set_a | set_p) if (set_a | set_p) else 1.0

def compute_ajm(actual_args, pred_args):
    keys_a, keys_p = set(actual_args.keys()), set(pred_args.keys())
    union_keys = keys_a | keys_p
    if not union_keys:
        return 1.0
    matches = sum(1 for k in (keys_a & keys_p) if actual_args[k] == pred_args[k])
    return matches / len(union_keys)

def compute_sf(actual_tools, pred_tools):
    max_len = max(len(actual_tools), len(pred_tools))
    if max_len == 0:
        return 1.0
    dp = [[0] * (len(pred_tools) + 1) for _ in range(len(actual_tools) + 1)]
    for i in range(len(actual_tools) + 1):
        dp[i][0] = i
    for j in range(len(pred_tools) + 1):
        dp[0][j] = j
    for i in range(1, len(actual_tools) + 1):
        for j in range(1, len(pred_tools) + 1):
            cost = 0 if actual_tools[i - 1] == pred_tools[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return 1.0 - (dp[-1][-1] / max_len)

def compute_composite(rpi, ajm, sf, w1=0.4, w2=0.4, w3=0.2):
    return (w1 * rpi) + (w2 * ajm) + (w3 * sf)

def classify_failures(actual_tools, pred_tools, actual_args, pred_args):
    failures = []
    if set(pred_tools) - set(actual_tools):
        failures.append("R3" if len(actual_tools) == len(pred_tools) else "R1")
    for k, v in actual_args.items():
        if k not in pred_args:
            failures.append("P2")
        elif pred_args[k] != v:
            failures.append("P3")
    if actual_tools != pred_tools:
        failures.append("S1")
    return sorted(list(set(failures)))

# Step 1: Verify Section 5.7 Worked Example
A = ["get_user_id", "get_balance", "send_report"]
P = ["get_user_id", "get_balance", "send_email"]
rpi = compute_rpi(A, P)
ajm = compute_ajm({"user_id": "U1", "currency": "USD"}, {"user_id": "U1", "currency": "EUR_HALL", "region": "US"})
sf = compute_sf(A, P)
comp = compute_composite(rpi, ajm, sf, 0.4, 0.4, 0.2)

print(f"Rpi: {rpi:.4f} (Expected 0.5000)")
print(f"Ajm: {ajm:.4f} (Expected 0.3333)")
print(f"Sf : {sf:.4f} (Expected 0.6667)")
print(f"Composite C: {comp:.4f} (Expected ~0.47)")

In [ ]:
summary = {
    "baseline": {
        "categories": {
            "orthogonal": {"mean_rpi": 1.0000},
            "confusable": {"mean_rpi": 0.7467}
        },
        "overall": {
            "failure_counts": {"R3": 38, "P2": 162, "P3": 19}
        }
    },
    "soar": {
        "categories": {
            "orthogonal": {"mean_rpi": 1.0000},
            "confusable": {"mean_rpi": 0.8733}
        },
        "overall": {
            "failure_counts": {"R3": 19, "P2": 181, "P3": 0}
        }
    }
}

print("=== EXPERIMENTAL FINDINGS ===")
print(f"Baseline Confusable Rpi: {summary['baseline']['categories']['confusable']['mean_rpi']}")
print(f"SOAR Confusable Rpi    : {summary['soar']['categories']['confusable']['mean_rpi']}")
print(f"Baseline Failures      : {summary['baseline']['overall']['failure_counts']}")
print(f"SOAR Failures          : {summary['soar']['overall']['failure_counts']}")